# My 2025 notebook for the NLP Competition on Disaster Tweets

## 1. Setup access to datasets

In [ ]:
from pathlib import Path
import pandas as pd

inputPath = Path("/kaggle/input/nlp-getting-started")
workingPath = Path("/kaggle/working/")

inputPath, workingPath

Check out what's in the raw data...

In [ ]:
raw_training_df = pd.read_csv(inputPath/'train.csv')
raw_training_df

# 2. Clean up datasets



Okay, so, let's remove a few columns we don't need. Let's create a function for doing this.

*Note: I'm assuming that the keyword and location information don't actually matter, when trying to figure out if a tweet's text is about a disaster or not.

In [ ]:
# NOTE: I refactored my code into a reusable function 
# Because cleaning the training and testing datasets will be the same.

def clean_up(df):
    result = df.drop(['keyword','location'],axis=1)
    result.rename(columns={'text': 'input', 'target': 'labels'},inplace=True)
    result['input'] = result['input'].fillna("[No text]")
    return result

In [ ]:
training_df = clean_up(raw_training_df)
training_df

Great! Next, we are going to use HF transformers, so we need to adapt the dataframes as datasets.

Annoyingly, FastAI doesn't have support for Transformers natively.

In [ ]:
!pip install datasets -q

In [ ]:
# we will use HuggingFace Transformers, so we need to convert the pandas dataframes into HF datasets.

from datasets import Dataset,DatasetDict

training_ds = Dataset.from_pandas(training_df)
training_ds

And let's try using the microsoft model variation of BERT. If you check out [HuggingFace Text Classification](https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads) you'll see a range of models that could be used for this competition.

*Note: I suggest using GPU accelerating, but select the P100 option!*

In [ ]:
model_name = 'microsoft/deberta-v3-xsmall'
from transformers import AutoModelForSequenceClassification,AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

Okay, so now we need to get at the tokens.

In [ ]:
tokenize_inputs = lambda x: tokenizer(x['input'])
token_ds = training_ds.map(tokenize_inputs, batched=True)
token_ds

So the dataset contains columns 'input_ids', 'token_type_ids', and 'attention_mask'. All of these where created by the tokenizer.

Next, let's have a look at one of these...

In [ ]:
row = token_ds[1]
row

So we can see that the input tweet has been converted into a set of numbers (the 'input_ids') - that's what we need for training!

Wonderful! Okay, moving on, we must split the token dataset into "train" and "test"

In [ ]:
split_ds = token_ds.train_test_split(0.25, seed=42)
split_ds

We also need a way to measure the performance, let's create a function for that. It needs to satisfy the competition guidelines - in this case that's using [F1 score](www.kaggle.com/competitions/nlp-getting-started/overview/evaluation).

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)  # For multi-class, or use >0.5 for binary
    return {
        "f1": f1_score(labels, predictions)
    }

Okay, so now we can create the trainer. We create a helper "training arguments" object first.

In [ ]:
from transformers import Trainer, TrainingArguments

bs = 128
epochs = 4

lr = 8e-5

args = TrainingArguments(
    output_dir='outputs',
    learning_rate=lr,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=True,
    eval_strategy="epoch",
    logging_strategy='epoch',
    logging_steps=50,
    save_strategy='epoch',
    per_device_train_batch_size=bs,
    per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none'
)

And now here is the Trainer itself - notice how it takes both the training part and testing part of the split token dataset. Notice also that the model is using 2 labels (1 - tweet is about a disaster, 0 - otherwise). 

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model, 
    args, 
    train_dataset=split_ds['train'], 
    eval_dataset=split_ds['test'],
    processing_class=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

And here is the evaluation dataset...

In [ ]:
raw_eval_df = pd.read_csv(inputPath/'test.csv')
eval_df = clean_up(raw_eval_df)
eval_df

In [ ]:
eval_ds = Dataset.from_pandas(eval_df).map(tokenize_inputs, batched=True)
eval_ds

In [ ]:
preds = trainer.predict(eval_ds)
preds

In [ ]:
logits = preds.predictions
logits.shape, logits

In [ ]:
import numpy as np
predicted_labels = np.argmax(logits, axis=1)  # picks the class with the higher logit
predicted_labels

In [ ]:
tweet_ids = eval_ds['id']

In [ ]:
import pandas as pd

submission = pd.DataFrame({
    'id': tweet_ids,
    'target': predicted_labels
})
assert len(submission) == len(eval_ds)

submission.to_csv("submission.csv", index=False)
submission